In [ ]:
# Importing libraries & frameworks 
import genesis as gs
import numpy as np
import cv2
from pathlib import Path


[I 04/10/25 15:25:46.676 4153405] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [ ]:
# Genesis initialization 
gs.init(
    backend=gs.cuda,
    seed=None,
    precision='32',
    debug=False,
    eps=1e-12,
    logging_level=None,
    theme='dark',
    logger_verbose_time=False
)

# Creating the scene
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, 0, 1.0),
        camera_lookat=(0, 0, 0.15),
        camera_fov=30,
        max_FPS=600
    ),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=False, 
    show_FPS=False
)

# Plane (ground) 
plane = scene.add_entity(gs.morphs.Plane())

# Bottle model
beer_bottle = scene.add_entity(
    gs.morphs.Mesh(
        file="/home/nexus/Desktop/Genesis/examples/Exp_RigidEntity/bottles/beer_bottle/beer_bottle.obj",  # Absolute or relative path
        pos=(0.0, 0.0, 0),            
        euler=(0, 0, 0.0),
        collision=True,
        visualization=True,
        fixed=True,          
        scale=0.01                      
    )
)

# Camera Setup
cam = scene.add_camera(
    pos    =(3, -1.5, 0.2),
    lookat = (0.0, 0.0, 0.09),
    res    = (1280, 960),
    fov    = 30,
    GUI    = False,
    up = (0, 0, 1),

)

# Building the Scene
scene.build()


[Genesis] [15:25:54] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [15:25:54] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [15:25:54] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [15:25:54] [INFO] Running on [NVIDIA RTX A2000 12GB] with backend gs.cuda. Device memory: 11.75 GB.
[Genesis] [15:25:54] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [15:25:55] [INFO] Scene <c4d6fc4> created.
[Genesis] [15:25:55] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <8d5de8c>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [15:25:55] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <73250b6>, morph: <gs.morphs.Mesh(file='/home/nexus/Desktop/Genesis/examples/Exp_RigidEntity/bottles/beer_bottle/beer_bottle.obj')>, material: <gs.materials.Rigid>.


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[Genesis] [15:25:56] [INFO] Building scene <c4d6fc4>...
[Genesis] [15:26:00] [INFO] Compiling simulation kernels...
[Genesis] [15:26:04] [INFO] Building visualizer...


In [4]:
# Output directory
output_dir = Path("../beer_bottle_dataset/")
output_dir.mkdir(parents=True, exist_ok=True)

# Number of samples
num_samples = 1  
data_metadata = []

In [6]:
# TOP VIEW: Getting RGB, Depth, Segmentation data's & Normal Map 
for i in range(num_samples):
    cam.set_pose(
    pos    = (0, 0, 1),
    lookat = (0, 0, 0.09),
    up = (0, 1, 0),
    )
    # Capture RGB-D data
    Top_render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False)
    Top_rgb, Top_depth, Top_seg, Top_normal_map = Top_render_output

    # Save data
    view_id = f"top_view_{i:02d}"
    np.save(output_dir / f"top_rgb.npy", Top_rgb)
    np.save(output_dir / f"top_depth.npy", Top_depth)
    np.save(output_dir / f"top_segmentation_mask.npy", Top_seg)
    np.save(output_dir / f"top_normal_map.npy", Top_normal_map)

    # Save RGB as PNG for visual verification
    cv2.imwrite(str(output_dir / "top_rgb.png"), cv2.cvtColor(Top_rgb, cv2.COLOR_RGB2BGR))
    
    # Save depth as PNG (scaled to 0-255)
    Top_depth_min, Top_depth_max = np.min(Top_depth), np.max(Top_depth)
    if Top_depth_max > Top_depth_min:  # Avoid division by zero
        Top_depth_scaled = (Top_depth - Top_depth_min) / (Top_depth_max - Top_depth_min) * 255
    else:
        Top_depth_scaled = np.zeros_like(Top_depth)
    
    Top_depth_scaled = Top_depth_scaled.astype(np.uint8)
    cv2.imwrite(str(output_dir / f"depth_{view_id}.png"), Top_depth_scaled)
    
    

In [7]:
# SIDE VIEW: Getting RGB, Depth, Segmentation data's & Normal Map 
for i in range(num_samples):
    cam.set_pose(
    pos    = (1, 0, 0.05),
    lookat = (0, 0, 0.09),
    up = (0, 0, 1),
    )

    # Capture RGB-D data
    side_render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False)
    side_rgb, side_depth, side_seg, side_normal_map = side_render_output
    
    # Save data
    view_id = f"side_view_{i:02d}"
    np.save(output_dir / f"side_rgb.npy", side_rgb)
    np.save(output_dir / f"side_depth.npy", side_depth)
    np.save(output_dir / f"side_segmentation_mask.npy", side_seg)
    np.save(output_dir / f"side_normal_map.npy", side_normal_map)
    
    # Save RGB as PNG for visual verification
    cv2.imwrite(str(output_dir / "side_rgb.png"), cv2.cvtColor(side_rgb, cv2.COLOR_RGB2BGR))
    
    # Save depth as PNG (scaled to 0-255)
    side_depth_min, side_depth_max = np.min(side_depth), np.max(side_depth)
    if side_depth_max > side_depth_min:  # Avoid division by zero
        side_depth_scaled = (side_depth - side_depth_min) / (side_depth_max - side_depth_min) * 255
    else:
        side_depth_scaled = np.zeros_like(side_depth)
    
    side_depth_scaled = side_depth_scaled.astype(np.uint8)
    cv2.imwrite(str(output_dir / f"depth_{view_id}.png"), side_depth_scaled)
    

In [8]:
# ISOMETRIC VIEW: Getting RGB, Depth, Segmentation data's & Normal Map 

for i in range(num_samples):
    cam.set_pose(
    pos    = (1, 1, 1),
    lookat = (0, 0, 0.09),
    up = (0, 0, 1),
    )

    # Capture RGB-D data
    iso_render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False)
    iso_rgb, iso_depth, iso_seg, iso_normal_map = iso_render_output
    
    # Save data
    view_id = f"iso_view_{i:02d}"
    np.save(output_dir / f"iso_rgb.npy", iso_rgb)
    np.save(output_dir / f"iso_depth.npy", iso_depth)
    np.save(output_dir / f"iso_segmentation_mask.npy", iso_seg)
    np.save(output_dir / f"iso_normal_map.npy", iso_normal_map)
    
    # Save RGB as PNG for visual verification
    cv2.imwrite(str(output_dir / "iso_rgb.png"), cv2.cvtColor(iso_rgb, cv2.COLOR_RGB2BGR))
    
    # Save depth as PNG (scaled to 0-255)
    iso_depth_min, iso_depth_max = np.min(iso_depth), np.max(iso_depth)
    if iso_depth_max > iso_depth_min:  # Avoid division by zero
        iso_depth_scaled = (iso_depth - iso_depth_min) / (iso_depth_max - iso_depth_min) * 255
    else:
        iso_depth_scaled = np.zeros_like(iso_depth)
    
    iso_depth_scaled = iso_depth_scaled.astype(np.uint8)
    cv2.imwrite(str(output_dir / f"depth_{view_id}.png"), iso_depth_scaled)